In [ ]:
print("Startup Job Finder – discovery layer ready")

Setup

In [3]:
from tavily import TavilyClient
from dotenv import load_dotenv
import os

load_dotenv()
client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

Company discovery query

In [4]:
query = (
  '"AI startup" '
  '("about us" OR "company" OR "who we are") '
  '-news -blog -article -wikipedia -linkedin -medium'
)

response = client.search(
    query=query,
    search_depth="advanced",
    max_results=20
)

for r in response["results"]:
    print("TITLE:", r["title"])
    print("URL:", r["url"])
    print("----")


TITLE: Artificial Intelligence, IT Company, AI Startup - Alpha AI
URL: https://alphaai.biz/our-story
----
TITLE: Our Company
URL: https://www.greeter.ai/company
----
TITLE: Day in the Life of an AI Startup Founder | YCombinator
URL: https://www.youtube.com/watch?v=Gt9dnFp1M0E
----
TITLE: How a hot AI startup got 800+ business customers by ...
URL: https://nextplayso.substack.com/p/how-a-hot-ai-startup-got-800-business
----
TITLE: AI NATION ABOUT US
URL: https://www.ai-nation.de/about-us
----
TITLE: The Impossible is Now Possible: AI Startup Showcase ...
URL: https://www.youtube.com/watch?v=hpwEYUvfSOg
----
TITLE: About Apera AI
URL: https://apera.ai/about-apera-ai/
----
TITLE: About Us - Google for Startups
URL: https://startup.google.com/about-us/
----
TITLE: About Us
URL: https://voice.ai/about
----
TITLE: List of interested Artificial Intelligence (AI) suppliers
URL: https://www.canada.ca/en/government/system/digital-government/digital-government-innovations/responsible-use-ai/list-

In [5]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

CAREER_KEYWORDS = [
    "career",
    "job",
    "join",
    "work with",
    "hiring",
]

BAD_HINTS = ["contact", "about", "privacy", "terms", "press", "blog"]

def looks_like_careers(text: str, href: str) -> bool:
    t = text.lower()
    h = href.lower()

    # reject anchors like "#Contact"
    if h.startswith("#"):
        return False

    # reject obvious non-careers pages
    if any(b in t or b in h for b in BAD_HINTS):
        return False

    # accept careers-ish
    return any(k in t or k in h for k in CAREER_KEYWORDS)


def find_careers_url(homepage: str):
    try:
        resp = requests.get(homepage, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        for a in soup.find_all("a", href=True):
            text = (a.get_text() or "").lower()
            href = a["href"].lower()

            if looks_like_careers(text, href):
                url = urljoin(homepage, href)
                return url.rstrip("/")

    except Exception as e:
        print("ERROR:", homepage, e)

    return None


In [6]:
companies = [
    "https://alphaai.biz",
    "https://ai-nation.de",
    "https://apera.ai",
]

for c in companies:
    print(c, "→", find_careers_url(c))


https://alphaai.biz → https://alphaai.biz/careers
https://ai-nation.de → None
https://apera.ai → https://apera.ai/careers


In [7]:
import requests
from bs4 import BeautifulSoup

def extract_careers_text(careers_url: str) -> str:
    resp = requests.get(careers_url, timeout=10)
    soup = BeautifulSoup(resp.text, "html.parser")

    # remove junk
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    text = soup.get_text(separator="\n")

    # basic cleanup
    lines = [line.strip() for line in text.splitlines()]
    lines = [l for l in lines if len(l) > 30]

    return "\n".join(lines[:200])  # cap: first ~200 lines


In [8]:
careers_pages = [
    ("Alpha AI", "https://alphaai.biz/careers"),
    ("Apera AI", "https://apera.ai/careers"),
]

for name, url in careers_pages:
    print("====", name, "====")
    text = extract_careers_text(url)
    print(text[:1000])
    print("\n")


==== Alpha AI ====
Alpha AI - Opportunities, Roles
Among India’s top 10 Companies in the Field of Applied Artificial Intelligence
Want to join our AI Research and Development team?
BENEFITS OF THE INTERNSHIPS / OPPORTUNITY
BENEFITS OF THE INTERNSHIPS / OPPORTUNITY
At the moment we are only accepting applications for
React Native / Flutter App Developer
Full Stack Web Developer - Intern
: Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.
: JavaScript, HTML, CSS, Python, PostgreSQL, Docker.
: Pursuing/recent CS degree, strong problem-solving skills.
React Native / Flutter App Developer - Intern
: Develop mobile apps with Flutter/React Native, offline AI models, and PostgreSQL/SQLite.
: Flutter/React Native, Python, PostgreSQL/SQLite, TensorFlow Lite, Docker.
: Pursuing/recent CS degree, passion for mobile development.
While we are a bootstrapped company and can offer a
, we value the opportunity for interns to gain hands-on expe

In [9]:
JOB_HINTS = [
    "engineer", "developer", "scientist", "researcher", "intern",
    "backend", "full stack", "full-stack", "machine learning", "ml", "ai"
]

def extract_jobish_lines(text: str, window: int = 4) -> str:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    keep = set()

    for i, line in enumerate(lines):
        low = line.lower()
        if any(h in low for h in JOB_HINTS):
            for j in range(max(0, i - window), min(len(lines), i + window + 1)):
                keep.add(j)

    out = [lines[i] for i in sorted(keep)]
    return "\n".join(out)


In [10]:
for name, url in careers_pages:
    print("====", name, "JOBISH ====")
    text = extract_careers_text(url)
    jobish = extract_jobish_lines(text, window=4)
    print(jobish[:1500])
    print("\n")


==== Alpha AI JOBISH ====
Alpha AI - Opportunities, Roles
Among India’s top 10 Companies in the Field of Applied Artificial Intelligence
Want to join our AI Research and Development team?
BENEFITS OF THE INTERNSHIPS / OPPORTUNITY
BENEFITS OF THE INTERNSHIPS / OPPORTUNITY
At the moment we are only accepting applications for
React Native / Flutter App Developer
Full Stack Web Developer - Intern
: Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.
: JavaScript, HTML, CSS, Python, PostgreSQL, Docker.
: Pursuing/recent CS degree, strong problem-solving skills.
React Native / Flutter App Developer - Intern
: Develop mobile apps with Flutter/React Native, offline AI models, and PostgreSQL/SQLite.
: Flutter/React Native, Python, PostgreSQL/SQLite, TensorFlow Lite, Docker.
: Pursuing/recent CS degree, passion for mobile development.
While we are a bootstrapped company and can offer a
, we value the opportunity for interns to gain hands-

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

def analyze_careers(company: str, text: str) -> str:
    prompt = f"""
You are analyzing startup careers pages for a job discovery tool.

Company: {company}

Rules (very important):
- ONLY extract roles that are explicitly mentioned in the TEXT.
- If you are not 100% sure a role exists in the TEXT, DO NOT include it.
- If there are no explicit roles, return an empty roles list but still provide overall_summary.
- Return a SINGLE valid JSON object only (no markdown, no code fences).
- A role title MUST be copied from a SINGLE line in TEXT that looks like a job title.
- Also return "title_evidence" which is exactly that line.
- If you can't find a job-title-like line, do not create a role.


Return JSON schema:
{{
  "relevant": boolean,
  "roles": [
    {{
      "title": string,
      "title_evidence": string,
      "seniority": "intern" | "junior" | "mid" | "senior" | "unknown",
      "tech_stack": [string],
      "remote_friendly": boolean | "unknown",
      "summary": string,
      "evidence": [string]
    }}
  ],
  "overall_summary": string
}}


TEXT:
{text}
"""
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content


In [19]:
alpha_text = extract_jobish_lines(
    extract_careers_text("https://alphaai.biz/careers")
)

print(analyze_careers("Alpha AI", alpha_text))


{
  "relevant": true,
  "roles": [
    {
      "title": "Full Stack Web Developer - Intern",
      "title_evidence": "Full Stack Web Developer - Intern",
      "seniority": "intern",
      "tech_stack": ["MEAN", "MERN", "Flask", "Django", "PostgreSQL", "pgvector", "offline LLMs", "JavaScript", "HTML", "CSS", "Python", "Docker"],
      "remote_friendly": "unknown",
      "summary": "Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.",
      "evidence": []
    },
    {
      "title": "React Native / Flutter App Developer - Intern",
      "title_evidence": "React Native / Flutter App Developer - Intern",
      "seniority": "intern",
      "tech_stack": ["Flutter", "React Native", "Python", "PostgreSQL", "SQLite", "TensorFlow Lite", "Docker"],
      "remote_friendly": "unknown",
      "summary": "Develop mobile apps with Flutter/React Native, offline AI models, and PostgreSQL/SQLite.",
      "evidence": []
    }
  ],
  "overall_sum

In [20]:
apera_text = extract_jobish_lines(
    extract_careers_text("https://apera.ai/careers")
)

print(analyze_careers("Apera AI", apera_text))


{
  "relevant": true,
  "roles": [
    {
      "title": "Principal Machine Learning Applied Scientist",
      "title_evidence": "Principal Machine Learning Applied Scientist, Kaggle Competitions Master and avid downhill skier",
      "seniority": "senior",
      "tech_stack": ["AI", "ML", "deep learning"],
      "remote_friendly": true,
      "summary": "Responsible for creating new inventions in artificial intelligence and deep learning, focusing on real-world results.",
      "evidence": [
        "Create new inventions in artificial intelligence and deep learning that will be used by the world's leading manufacturers.",
        "Problem solving ability is really important."
      ]
    },
    {
      "title": "Principal Software Development Engineer",
      "title_evidence": "Principal Software Development Engineer, devoted dad and hockey player",
      "seniority": "senior",
      "tech_stack": ["software engineering"],
      "remote_friendly": true,
      "summary": "Involved in s

In [21]:
import json

def analyze_careers_json(company: str, text: str) -> dict:
    raw = analyze_careers(company, text)
    return json.loads(raw)   

In [22]:
alpha = analyze_careers_json("Alpha AI", alpha_text)
apera = analyze_careers_json("Apera AI", apera_text)

alpha, apera


({'relevant': True,
  'roles': [{'title': 'Full Stack Web Developer - Intern',
    'title_evidence': 'Full Stack Web Developer - Intern',
    'seniority': 'intern',
    'tech_stack': ['MEAN',
     'MERN',
     'Flask',
     'Django',
     'PostgreSQL',
     'pgvector',
     'offline LLMs',
     'JavaScript',
     'HTML',
     'CSS',
     'Python',
     'Docker'],
    'remote_friendly': 'unknown',
    'summary': 'Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.',
    'evidence': ['Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.',
     'Pursuing/recent CS degree, strong problem-solving skills.']},
   {'title': 'React Native / Flutter App Developer - Intern',
    'title_evidence': 'React Native / Flutter App Developer - Intern',
    'seniority': 'intern',
    'tech_stack': ['Flutter',
     'React Native',
     'Python',
     'PostgreSQL',
     'SQLite',
     'TensorFl

In [23]:
TITLE_HINTS = ["engineer", "developer", "scientist", "intern", "manager", "lead", "principal", "research"]

def extract_title_lines(text: str) -> list[str]:
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    out = []
    for l in lines:
        low = l.lower()
        # rövid, cím-szerű sorok preferálása
        if any(h in low for h in TITLE_HINTS) and len(l) <= 120:
            out.append(l)
    return out[:30]


In [24]:
titles = extract_title_lines(apera_text)
apera_input = apera_text + "\n\nPOSSIBLE TITLES:\n" + "\n".join(titles)

apera = analyze_careers_json("Apera AI", apera_input)
apera


{'relevant': True,
 'roles': [{'title': 'Principal Machine Learning Applied Scientist',
   'title_evidence': 'Principal Machine Learning Applied Scientist, Kaggle Competitions Master and avid downhill skier',
   'seniority': 'unknown',
   'tech_stack': ['AI', 'ML', 'deep learning'],
   'remote_friendly': True,
   'summary': 'Responsible for creating new inventions in artificial intelligence and deep learning, focusing on achieving real-world results.',
   'evidence': ["Create new inventions in artificial intelligence and deep learning that will be used by the world's leading manufacturers.",
    'Problem solving ability is really important.']},
  {'title': 'Principal Software Development Engineer',
   'title_evidence': 'Principal Software Development Engineer, devoted dad and hockey player',
   'seniority': 'unknown',
   'tech_stack': ['software engineering', 'cloud computing'],
   'remote_friendly': True,
   'summary': 'Involved in software development with a focus on delivering best-

In [25]:
def flatten_company_result(company_name: str, careers_url: str, data: dict) -> list[dict]:
    rows = []
    for r in data.get("roles", []):
        rows.append({
            "company": company_name,
            "careers_url": careers_url,
            "title": r.get("title"),
            "seniority": r.get("seniority", "unknown"),
            "remote_friendly": r.get("remote_friendly", "unknown"),
            "tech_stack": ", ".join(r.get("tech_stack", [])),
            "summary": r.get("summary", ""),
            "evidence": " | ".join(r.get("evidence", [])[:2]),
        })    
    if not rows:
        rows.append({
            "company": company_name,
            "careers_url": careers_url,
            "title": None,
            "seniority": None,
            "remote_friendly": None,
            "tech_stack": "",
            "summary": data.get("overall_summary", ""),
            "evidence": "",
        })
    return rows


In [26]:
alpha_url = "https://alphaai.biz/careers"
apera_url = "https://apera.ai/careers"

rows = []
rows += flatten_company_result("Alpha AI", alpha_url, alpha)
rows += flatten_company_result("Apera AI", apera_url, apera)

rows


[{'company': 'Alpha AI',
  'careers_url': 'https://alphaai.biz/careers',
  'title': 'Full Stack Web Developer - Intern',
  'seniority': 'intern',
  'remote_friendly': 'unknown',
  'tech_stack': 'MEAN, MERN, Flask, Django, PostgreSQL, pgvector, offline LLMs, JavaScript, HTML, CSS, Python, Docker',
  'summary': 'Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.',
  'evidence': 'Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM. | Pursuing/recent CS degree, strong problem-solving skills.'},
 {'company': 'Alpha AI',
  'careers_url': 'https://alphaai.biz/careers',
  'title': 'React Native / Flutter App Developer - Intern',
  'seniority': 'intern',
  'remote_friendly': 'unknown',
  'tech_stack': 'Flutter, React Native, Python, PostgreSQL, SQLite, TensorFlow Lite, Docker',
  'summary': 'Develop mobile apps with Flutter/React Native, offline AI models, and PostgreSQL/SQLite.',

In [27]:
import pandas as pd

df = pd.DataFrame(rows)
df


,company,careers_url,title,seniority,remote_friendly,tech_stack,summary,evidence
0,Alpha AI,https://alphaai.biz/careers,Full Stack Web Developer - Intern,intern,unknown,"MEAN, MERN, Flask, Django, PostgreSQL, pgvecto...","Build web apps with MEAN/MERN stack, APIs (Fla...","Build web apps with MEAN/MERN stack, APIs (Fla..."
1,Alpha AI,https://alphaai.biz/careers,React Native / Flutter App Developer - Intern,intern,unknown,"Flutter, React Native, Python, PostgreSQL, SQL...","Develop mobile apps with Flutter/React Native,...","Develop mobile apps with Flutter/React Native,..."
2,Apera AI,https://apera.ai/careers,Principal Machine Learning Applied Scientist,unknown,True,"AI, ML, deep learning",Responsible for creating new inventions in art...,Create new inventions in artificial intelligen...
3,Apera AI,https://apera.ai/careers,Principal Software Development Engineer,unknown,True,"software engineering, cloud computing",Involved in software development with a focus ...,Apera AI is a technically rich environment. | ...


In [28]:
from urllib.parse import urlparse

BLOCKED_DOMAINS = [
    "linkedin.com",
    "indeed.com",
    "ziprecruiter.com",
    "randstad",
    "crossover.com",
    "amazon.jobs",
    "youtube.com",
    "wikipedia.org",
    "nytimes.com",
    "aws.amazon.com",
    "un.org",
    "unesco.org",
    "nasa.gov",
    "nsf.gov",
    "substack.com",
    "medium.com",
]

BLOCKED_PATH_HINTS = [
    "/blog/",
    "/news/",
    "/article",
    "/posts/",
    "/journal/",
]

def is_noise_url(url: str) -> bool:
    u = url.lower()
    domain = urlparse(u).netloc
    path = urlparse(u).path
    if any(d in domain for d in BLOCKED_DOMAINS):
        return True
    if any(h in path for h in BLOCKED_PATH_HINTS):
        return True
    return False

def root_homepage(url: str) -> str:
    p = urlparse(url)
    return f"{p.scheme}://{p.netloc}"

# Tavily találatok -> homepages
homepages = []
seen = set()
for r in response["results"]:
    url = r["url"]
    if is_noise_url(url):
        continue
    hp = root_homepage(url)
    if hp in seen:
        continue
    seen.add(hp)
    homepages.append({
        "title": r.get("title", ""),
        "source_url": url,
        "homepage": hp
    })

len(homepages), homepages[:5]


(14,
 [{'title': 'Artificial Intelligence, IT Company, AI Startup - Alpha AI',
   'source_url': 'https://alphaai.biz/our-story',
   'homepage': 'https://alphaai.biz'},
  {'title': 'Our Company',
   'source_url': 'https://www.greeter.ai/company',
   'homepage': 'https://www.greeter.ai'},
  {'title': 'AI NATION ABOUT US',
   'source_url': 'https://www.ai-nation.de/about-us',
   'homepage': 'https://www.ai-nation.de'},
  {'title': 'About Apera AI',
   'source_url': 'https://apera.ai/about-apera-ai/',
   'homepage': 'https://apera.ai'},
  {'title': 'About Us - Google for Startups',
   'source_url': 'https://startup.google.com/about-us/',
   'homepage': 'https://startup.google.com'}])

In [29]:
N = 10  
for item in homepages[:N]:
    item["careers_url"] = find_careers_url(item["homepage"])

[item for item in homepages[:N]]


[{'title': 'Artificial Intelligence, IT Company, AI Startup - Alpha AI',
  'source_url': 'https://alphaai.biz/our-story',
  'homepage': 'https://alphaai.biz',
  'careers_url': 'https://alphaai.biz/careers'},
 {'title': 'Our Company',
  'source_url': 'https://www.greeter.ai/company',
  'homepage': 'https://www.greeter.ai',
  'careers_url': 'https://www.greeter.ai/careers'},
 {'title': 'AI NATION ABOUT US',
  'source_url': 'https://www.ai-nation.de/about-us',
  'homepage': 'https://www.ai-nation.de',
  'careers_url': None},
 {'title': 'About Apera AI',
  'source_url': 'https://apera.ai/about-apera-ai/',
  'homepage': 'https://apera.ai',
  'careers_url': 'https://apera.ai/careers'},
 {'title': 'About Us - Google for Startups',
  'source_url': 'https://startup.google.com/about-us/',
  'homepage': 'https://startup.google.com',
  'careers_url': 'https://startup.google.com/community'},
 {'title': 'About Us',
  'source_url': 'https://voice.ai/about',
  'homepage': 'https://voice.ai',
  'career

In [30]:
def safe_extract_jobish(careers_url: str) -> str | None:
    if not careers_url:
        return None
    try:
        raw = extract_careers_text(careers_url)
        jobish = extract_jobish_lines(raw, window=4)
        return jobish.strip() if jobish.strip() else None
    except Exception as e:
        return None

for item in homepages[:N]:
    item["jobish_text"] = safe_extract_jobish(item.get("careers_url"))

[(x["homepage"], x["careers_url"], bool(x["jobish_text"])) for x in homepages[:N]]


[('https://alphaai.biz', 'https://alphaai.biz/careers', True),
 ('https://www.greeter.ai', 'https://www.greeter.ai/careers', True),
 ('https://www.ai-nation.de', None, False),
 ('https://apera.ai', 'https://apera.ai/careers', True),
 ('https://startup.google.com', 'https://startup.google.com/community', True),
 ('https://voice.ai', 'https://voice.ai/jobs', False),
 ('https://www.canada.ca', None, False),
 ('https://www.alanany.com', None, False),
 ('https://www.heroai.ca', 'https://www.heroai.ca', True),
 ('https://www.elits.com', 'https://www.elits.com/career', True)]

In [31]:
MAX_LLM = 5
llm_done = 0

for item in homepages[:N]:
    if llm_done >= MAX_LLM:
        break

    jt = item.get("jobish_text")
    if not jt:
        item["llm"] = None
        continue

    company_name = item["homepage"].replace("https://", "").replace("http://", "")
    item["llm"] = analyze_careers_json(company_name, jt)
    llm_done += 1

llm_done


5

In [32]:
rows = []

for item in homepages[:N]:
    if not item.get("llm"):
        continue
    company_name = item["homepage"].replace("https://", "").replace("http://", "")
    rows += flatten_company_result(company_name, item.get("careers_url"), item["llm"])

rows[:5], len(rows)


([{'company': 'alphaai.biz',
   'careers_url': 'https://alphaai.biz/careers',
   'title': 'Full Stack Web Developer - Intern',
   'seniority': 'intern',
   'remote_friendly': 'unknown',
   'tech_stack': 'MEAN, MERN, Flask, Django, PostgreSQL, pgvector, offline LLMs, JavaScript, HTML, CSS, Python, Docker',
   'summary': 'Build web apps with MEAN/MERN stack, APIs (Flask/Django), PostgreSQL (pgvector), and offline LLMs via WebLLM.',
   'evidence': ''},
  {'company': 'alphaai.biz',
   'careers_url': 'https://alphaai.biz/careers',
   'title': 'React Native / Flutter App Developer - Intern',
   'seniority': 'intern',
   'remote_friendly': 'unknown',
   'tech_stack': 'Flutter, React Native, Python, PostgreSQL, SQLite, TensorFlow Lite, Docker',
   'summary': 'Develop mobile apps with Flutter/React Native, offline AI models, and PostgreSQL/SQLite.',
   'evidence': ''},
  {'company': 'www.greeter.ai',
   'careers_url': 'https://www.greeter.ai/careers',
   'title': 'software engineering',
   'sen

In [33]:
import pandas as pd
df = pd.DataFrame(rows)
df


,company,careers_url,title,seniority,remote_friendly,tech_stack,summary,evidence
0,alphaai.biz,https://alphaai.biz/careers,Full Stack Web Developer - Intern,intern,unknown,"MEAN, MERN, Flask, Django, PostgreSQL, pgvecto...","Build web apps with MEAN/MERN stack, APIs (Fla...",
1,alphaai.biz,https://alphaai.biz/careers,React Native / Flutter App Developer - Intern,intern,unknown,"Flutter, React Native, Python, PostgreSQL, SQL...","Develop mobile apps with Flutter/React Native,...",
2,www.greeter.ai,https://www.greeter.ai/careers,software engineering,unknown,unknown,"iOS, back-end",Help us tackle the world's most pressing problem.,We're growing! Help us improve Workplace Healt...
3,apera.ai,https://apera.ai/careers,Principal Machine Learning Applied Scientist,senior,True,"AI, ML, deep learning",Responsible for creating new inventions in art...,Create new inventions in artificial intelligen...
4,apera.ai,https://apera.ai/careers,Principal Software Development Engineer,senior,True,"software engineering, cloud computing",Involved in software development with a focus ...,Apera AI is a technically rich environment. | ...
5,startup.google.com,https://startup.google.com/community,None,None,None,,The text describes the Google for Startups alu...,
6,www.heroai.ca,https://www.heroai.ca,None,None,None,,The text does not explicitly mention any job r...,


In [34]:
import json, time

out = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "query": query,
    "N": N,
    "items": homepages[:N],
    "rows": rows
}

with open("notebooks/run_output.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

"saved notebooks/run_output.json"


FileNotFoundError: [Errno 2] No such file or directory: 'notebooks/run_output.json'